# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core Segurança & Diagnóstico

Este notebook integra todos os componentes desenvolvidos no **Módulo 1**: telemetria de sensores, discretização proposicional, prova de tautologias de segurança e motor especialista de inferência aplicado à Máquina de Envasamento de Copos.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Any

class MapeadorProposicional:
    def extrair_proposicoes(self, telemetria: Dict[str, float]) -> Dict[str, bool]:
        tem_copo = bool(telemetria.get('Sensor_Capacitivo', 0))
        t1_atingida = telemetria.get('Temp_Termosselagem', 0.0) >= 180.0
        return {
            'tem_copo': tem_copo,
            'falta_copo': not tem_copo,
            't1': t1_atingida,
            'baixa_temp': not t1_atingida,
            'e1': bool(telemetria.get('Botao_Emergencia', 0)),
            'mesa_posicionada': bool(telemetria.get('Sensor_Posicao_Mesa', 1)),
        }

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []
    def adicionar_regra(self, id_r, antecedentes, consequente, desc, prioridade=1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento):
        self.bc = base_conhecimento
    def forward_chaining(self, fatos_iniciais):
        fatos_conhecidos = set(fatos_iniciais)
        historico = []
        passo = 1
        novos = True
        while novos:
            novos = False
            for regra in sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True):
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico.append({"Passo": passo, "Regra": regra.id_regra, "Diagnóstico": regra.descricao_diagnostico})
                    passo += 1
                    novos = True
                    break
        return fatos_conhecidos, historico

print("[OK] Classes Base do SCADA-Core carregadas!")


[OK] Classes Base do SCADA-Core carregadas!


In [2]:
class SCADACoreEnvasadora:
    def __init__(self):
        self.mapeador = MapeadorProposicional()
        self.bc = BaseConhecimento()

        self.bc.adicionar_regra("R-01", ["e1"], "parada_imediata", "Botão de Emergência Acionado", 10)
        self.bc.adicionar_regra("R-02", ["falta_copo"], "alarme_sem_copo", "Falta de Copo na Dispensação", 9)
        self.bc.adicionar_regra("R-03", ["alarme_sem_copo"], "bloqueia_dosador", "Bico Bloqueado (Cascata)", 8)
        self.bc.adicionar_regra("R-04", ["tem_copo", "baixa_temp"], "bloqueia_prensa", "Bloqueio Térmico da Selagem", 8)
        self.bc.adicionar_regra("R-05", ["tem_copo", "t1"], "libera_prensa", "Condições Ideais para Selagem", 5)

        self.motor = MotorInferencia(self.bc)

    def processar_ciclo_scan(self, telemetria: Dict[str, float]) -> Dict[str, Any]:
        props = self.mapeador.extrair_proposicoes(telemetria)
        fatos_ativos = {k for k, v in props.items() if v}

        trip = props['e1'] or props['falta_copo'] or (props['tem_copo'] and props['baixa_temp'])

        fatos_inf, trilha = self.motor.forward_chaining(fatos_ativos)
        return {
            "Trip_Ativo": trip,
            "Diagnósticos": list(fatos_inf),
            "Trilha": trilha
        }

core_maquina = SCADACoreEnvasadora()

res_a = core_maquina.processar_ciclo_scan({'Sensor_Capacitivo': 0.0, 'Temp_Termosselagem': 185.0, 'Botao_Emergencia': 0.0})
res_b = core_maquina.processar_ciclo_scan({'Sensor_Capacitivo': 1.0, 'Temp_Termosselagem': 160.0, 'Botao_Emergencia': 0.0})
res_c = core_maquina.processar_ciclo_scan({'Sensor_Capacitivo': 1.0, 'Temp_Termosselagem': 182.0, 'Botao_Emergencia': 0.0})

print("=== AVALIAÇÃO DE INTERTRAVAMENTOS (MÓDULO 1) ===")
print(f"-> Cenário A (Falta Copo) | Trip: {res_a['Trip_Ativo']} | Infere: {'bloqueia_dosador' in res_a['Diagnósticos']}")
print(f"-> Cenário B (Frio)       | Trip: {res_b['Trip_Ativo']} | Infere: {'bloqueia_prensa' in res_b['Diagnósticos']}")
print(f"-> Cenário C (Normal)     | Trip: {res_c['Trip_Ativo']} | Infere: {'libera_prensa' in res_c['Diagnósticos']}")

assert res_a["Trip_Ativo"] is True
assert "alarme_sem_copo" in res_a["Diagnósticos"]
assert "bloqueia_dosador" in res_a["Diagnósticos"]

assert res_b["Trip_Ativo"] is True
assert "bloqueia_prensa" in res_b["Diagnósticos"]

assert res_c["Trip_Ativo"] is False
assert "libera_prensa" in res_c["Diagnósticos"]

print("\n[OK] Avaliação Módulo 1 concluída com 100% de sucesso na Envasadora!")


=== AVALIAÇÃO DE INTERTRAVAMENTOS (MÓDULO 1) ===
-> Cenário A (Falta Copo) | Trip: True | Infere: True
-> Cenário B (Frio)       | Trip: True | Infere: True
-> Cenário C (Normal)     | Trip: False | Infere: True

[OK] Avaliação Módulo 1 concluída com 100% de sucesso na Envasadora!
